# 1. Imports

In [ ]:
import json, os, random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, Counter

: 

# 2. Config

In [ ]:
GRAPHDATA_PATH = r'C:\users\anonymous\Downloads\DeepTraLog-main (1)\DeepTraLog-main\GraphData'
RESULTS_PATH = r'C:\Users\Anonymous\OneDrive\Desktop\train-ticket\2026-mcm-cullime2-hadrian2\src\self_healing_agent\src\rl_models_2\results'
SAVE_PATH_A2 = RESULTS_PATH + r'\\A2_reinforce_scratch.pt'
SAVE_PATH_A3 = RESULTS_PATH + r'\\A3_reinforce_pretrain.pt'
import os; os.makedirs(RESULTS_PATH, exist_ok=True)
 
# Training hyperparameters
A2_EPISODES     = 1000
A2_LR           = 1e-3
A3_PRETRAIN_EP  = 30
A3_PRETRAIN_LR  = 1e-3
A3_PRETRAIN_BS  = 32
A3_FINETUNE_EP  = 2000
A3_FINETUNE_LR  = 1e-4
A3_ENTROPY      = 0.01
SEED            = 42
NOISE        = 0.3

ACTIONS = [
    "RESTART",
    "SCALE_UP",
    "REROUTE",
    "ROLLBACK",
]

ACTION2IDX = {
    action: index
    for index, action in enumerate(ACTIONS)
}

GROUND_TRUTH = {
    "F01": "ROLLBACK",
    "F02": "ROLLBACK",
    "F03": "SCALE_UP",
    "F04": "RESTART",
    "F05": "SCALE_UP",

    # F06 previously used CIRCUIT_BREAK. With the reduced action
    # space, RESTART is the closest supported recovery.
    "F06": "RESTART",

    "F07": "REROUTE",
    "F08": "RESTART",
    "F09": "ROLLBACK",
    "F10": "ROLLBACK",
    "F11": "ROLLBACK",
    "F12": "ROLLBACK",
    "F13": "ROLLBACK",
    "F14": "RESTART",

    # F23 previously used CIRCUIT_BREAK.
    "F23": "RESTART",

    "F24": "REROUTE",
    "F25": "SCALE_UP",
    "unknown": "ROLLBACK",
}

WINDOW = 100  # rolling window for convergence charts

#  3. Shared Helpers

In [ ]:
def get_fault_category(fault_type):
    return fault_type.split('-')[0] if '-' in fault_type else fault_type
 
def build_state(category, score=0.8, noise_level=0.0):
    state = [
        score,
        1.0 if category in ("F04","F08","F14") else 0.0,
        1.0 if category in ("F06","F07","F23") else 0.0,
        1.0 if category in ("F03","F05","F25") else 0.0,
        1.0 if category in ("F07","F24")        else 0.0,
        1.0,
        1.0 if category in ("F08","F22")        else 0.0,  # db error
        1.0 if category in ("F14","F10","F11","F12","F13") else 0.0,  # logic
        1.0 if category in ("F01","F02","F09")  else 0.0,  # async
        1.0 if category in ("F06","F08","F22")  else 0.0,  # err ratio proxy
    ]
    if noise_level > 0:
        noise = np.random.normal(0, noise_level, len(state))
        state = np.clip(np.array(state) + noise, 0.0, 1.0).tolist()
    return state
 
def load_all_traces(path):
    traces = []
    for fname in sorted(f for f in os.listdir(path) if f.endswith('.jsons')):
        with open(os.path.join(path, fname), 'r') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    g = json.loads(line)
                    if not g['trace_bool']:
                        cat = get_fault_category(g.get('error_trace_type','unknown'))
                        if cat in GROUND_TRUTH:
                            traces.append({
                                'category': cat,
                                'correct':  GROUND_TRUTH[cat],
                                'label':    ACTION2IDX[GROUND_TRUTH[cat]],
                            })
                except: continue
    return traces
 

# 4. Load Data

In [ ]:
print(f"Loading GraphData from {GRAPHDATA_PATH}...")
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
 
all_traces = load_all_traces(GRAPHDATA_PATH)
print(f"Total anomalous traces: {len(all_traces)}")
 
dist = Counter(t['category'] for t in all_traces)
print(f"\nFault distribution:")
for cat, count in sorted(dist.items()):
    print(f"  {cat}: {count} traces")
 
random.shuffle(all_traces)
split      = int(len(all_traces) * 0.8)
train_data = all_traces[:split]
test_data  = all_traces[split:]
print(f"\nTrain: {len(train_data)}  Test: {len(test_data)}")

# 5. Evaluation Helper

In [ ]:
def evaluate_agent(predict_fn, traces, label, noise_level=0.0):
    """Generic evaluation for any agent."""
    results       = []
    fault_correct = defaultdict(int)
    fault_counts  = defaultdict(int)
    action_counts = defaultdict(int)
 
    for trace in traces:
        cat        = trace['category']
        correct    = trace['correct']
        score      = round(random.uniform(0.5, 1.0), 3)
        state      = build_state(cat, score, noise_level)
        action     = predict_fn(state, cat)
        is_correct = (action == correct)
        results.append(is_correct)
        fault_correct[cat] += int(is_correct)
        fault_counts[cat]  += 1
        action_counts[action] += 1
 
    accuracy = np.mean(results) * 100
 
    print(f"\n{'='*55}")
    print(f"EVALUATION — {label}")
    print(f"{'='*55}")
    print(f"Accuracy: {accuracy:.1f}%")
    print(f"\n{'Fault':6s} {'N':6s} {'Acc%':7s} {'Expected'}")
    print(f"{'─'*40}")
    for cat in sorted(fault_counts):
        n   = fault_counts[cat]
        c   = fault_correct[cat]
        acc = c/n*100
        exp = GROUND_TRUTH.get(cat,'?')
        flag = "[GOOD]" if acc>=90 else "[FAIR]" if acc>=70 else "[POOR]"
        print(f"  {flag} {cat:5s}  {n:5d}  {acc:6.1f}%  {exp}")
 
    return {
        "label":     label,
        "accuracy":  accuracy,
        "per_fault": {c: fault_correct[c]/fault_counts[c]*100
                      for c in fault_counts},
    }
 
print("Evaluation helper loaded")

# 6. A1: RULE-BASED AGENT
No training needed — just hardcoded rules

In [ ]:
print("\n" + "="*55)
print("A1: RULE-BASED AGENT")
print("="*55)
 
def a1_predict(state, category=None):
    """
    A1: hardcoded threshold rules on state features.
    Does NOT use fault category directly — uses noisy
    state vector same as A2/A3.
    """
    score   = state[0]
    crash   = state[1]
    slow    = state[2]
    cpu     = state[3]
    network = state[4]

    if cpu > 0.5:
        return "SCALE_UP"

    # Network degradation must be checked before generic latency,
    # because a network fault can also set slow_response.
    if network > 0.5:
        return "REROUTE"

    if crash > 0.5 or slow > 0.5:
        return "RESTART"

    return "ROLLBACK"
 
a1_results = evaluate_agent(a1_predict, test_data, "A1 Rule-Based", noise_level=NOISE)

# 7. A2: REINFORCE FROM SCRATCH

In [ ]:
print("\n" + "="*55)
print("A2: REINFORCE FROM SCRATCH")
print("="*55)


class PolicyNetA2(nn.Module):
    def __init__(
        self,
        state_dim=10,
        action_dim=None,
    ):
        super().__init__()

        if action_dim is None:
            action_dim = len(ACTIONS)

        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return torch.softmax(
            self.net(x),
            dim=-1,
        )

    def logits(self, x):
        return self.net(x)


torch.manual_seed(SEED)

policy_a2 = PolicyNetA2(
    action_dim=len(ACTIONS),
)

optimizer = torch.optim.Adam(
    policy_a2.parameters(),
    lr=A2_LR,
)

print(
    f"Training {A2_EPISODES} episodes "
    f"with actions={ACTIONS}..."
)

ep_correct_a2 = []
ep_rewards_a2 = []

policy_a2.train()

for ep in range(A2_EPISODES):
    trace = random.choice(train_data)
    category = trace["category"]
    correct_action = trace["correct"]

    score = round(
        random.uniform(0.5, 1.0),
        3,
    )

    state_t = torch.tensor(
        build_state(
            category,
            score,
            noise_level=NOISE,
        ),
        dtype=torch.float,
    )

    probabilities = policy_a2(state_t)
    distribution = torch.distributions.Categorical(
        probabilities
    )

    action_index = distribution.sample()
    action = ACTIONS[action_index.item()]

    reward = (
        10.0
        if action == correct_action
        else -5.0
    )

    loss = (
        -distribution.log_prob(action_index)
        * reward
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    ep_correct_a2.append(
        int(action == correct_action)
    )
    ep_rewards_a2.append(reward)

    if (ep + 1) % 200 == 0:
        accuracy = (
            np.mean(ep_correct_a2[-200:])
            * 100
        )
        average_reward = np.mean(
            ep_rewards_a2[-200:]
        )

        print(
            f"  Episode {ep + 1:5d}: "
            f"acc={accuracy:.1f}%  "
            f"avg_reward={average_reward:.2f}"
        )


def a2_predict(
    state,
    category=None,
):
    policy_a2.eval()

    with torch.no_grad():
        probabilities = policy_a2(
            torch.tensor(
                state,
                dtype=torch.float,
            )
        )

    return ACTIONS[
        probabilities.argmax().item()
    ]


a2_results = evaluate_agent(
    a2_predict,
    test_data,
    "A2 REINFORCE Scratch",
    noise_level=NOISE,
)

torch.save(
    {
        "model_state": policy_a2.state_dict(),
        "actions": ACTIONS,
        "ground_truth": GROUND_TRUTH,
        "results": a2_results,
        "state_dim": 10,
        "action_dim": len(ACTIONS),
    },
    SAVE_PATH_A2,
)

print(f"A2 saved to {SAVE_PATH_A2}")


# A3: REINFORCE + SUPERVISED PRETRAINING

In [ ]:
print("\n" + "="*55)
print("A3: REINFORCE + SUPERVISED PRETRAINING")
print("="*55)


class PolicyNetA3(nn.Module):
    def __init__(
        self,
        state_dim=10,
        action_dim=None,
    ):
        super().__init__()

        if action_dim is None:
            action_dim = len(ACTIONS)

        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return torch.softmax(
            self.net(x),
            dim=-1,
        )

    def logits(self, x):
        return self.net(x)


torch.manual_seed(SEED)

policy_a3 = PolicyNetA3(
    action_dim=len(ACTIONS),
)


# ----------------------------------------------------------
# Phase 1: supervised pretraining
# ----------------------------------------------------------
print(
    f"\nPhase 1: Supervised pretraining "
    f"({A3_PRETRAIN_EP} epochs)..."
)

optimizer_pre = torch.optim.Adam(
    policy_a3.parameters(),
    lr=A3_PRETRAIN_LR,
)

criterion = nn.CrossEntropyLoss()

policy_a3.train()

for epoch in range(A3_PRETRAIN_EP):
    random.shuffle(train_data)

    epoch_loss = 0.0
    epoch_correct = 0

    for index in range(
        0,
        len(train_data),
        A3_PRETRAIN_BS,
    ):
        batch = train_data[
            index:index + A3_PRETRAIN_BS
        ]

        states = torch.tensor(
            [
                build_state(
                    trace["category"],
                    round(
                        random.uniform(0.5, 1.0),
                        3,
                    ),
                    NOISE,
                )
                for trace in batch
            ],
            dtype=torch.float,
        )

        labels = torch.tensor(
            [
                trace["label"]
                for trace in batch
            ],
            dtype=torch.long,
        )

        logits = policy_a3.logits(states)
        loss = criterion(
            logits,
            labels,
        )

        optimizer_pre.zero_grad()
        loss.backward()
        optimizer_pre.step()

        epoch_loss += (
            loss.item()
            * len(batch)
        )

        epoch_correct += (
            logits.argmax(dim=1)
            == labels
        ).sum().item()

    if (epoch + 1) % 10 == 0:
        accuracy = (
            epoch_correct
            / len(train_data)
            * 100
        )

        print(
            f"  Epoch {epoch + 1:3d}/"
            f"{A3_PRETRAIN_EP}: "
            f"loss={epoch_loss / len(train_data):.4f}  "
            f"acc={accuracy:.1f}%"
        )


# ----------------------------------------------------------
# Phase 2: REINFORCE fine-tuning
# ----------------------------------------------------------
print(
    f"\nPhase 2: REINFORCE fine-tuning "
    f"({A3_FINETUNE_EP} episodes)..."
)

optimizer_rl = torch.optim.Adam(
    policy_a3.parameters(),
    lr=A3_FINETUNE_LR,
)

baseline_value = 0.0
baseline_alpha = 0.1

ep_correct_a3 = []
ep_rewards_a3 = []

policy_a3.train()

for ep in range(A3_FINETUNE_EP):
    trace = random.choice(train_data)
    category = trace["category"]
    correct_action = trace["correct"]

    score = round(
        random.uniform(0.5, 1.0),
        3,
    )

    state_t = torch.tensor(
        build_state(
            category,
            score,
            NOISE,
        ),
        dtype=torch.float,
    )

    probabilities = policy_a3(state_t)

    distribution = (
        torch.distributions.Categorical(
            probabilities
        )
    )

    action_index = distribution.sample()
    action = ACTIONS[action_index.item()]

    reward = (
        10.0
        if action == correct_action
        else -5.0
    )

    advantage = reward - baseline_value

    baseline_value += (
        baseline_alpha
        * (reward - baseline_value)
    )

    entropy = -(
        probabilities
        * torch.log(
            probabilities + 1e-8
        )
    ).sum()

    loss = (
        -distribution.log_prob(action_index)
        * advantage
        - A3_ENTROPY * entropy
    )

    optimizer_rl.zero_grad()
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        policy_a3.parameters(),
        1.0,
    )

    optimizer_rl.step()

    ep_correct_a3.append(
        int(action == correct_action)
    )
    ep_rewards_a3.append(reward)

    if (ep + 1) % 500 == 0:
        accuracy = (
            np.mean(ep_correct_a3[-500:])
            * 100
        )

        average_reward = np.mean(
            ep_rewards_a3[-500:]
        )

        print(
            f"  Episode {ep + 1:5d}: "
            f"acc={accuracy:.1f}%  "
            f"avg_reward={average_reward:.2f}  "
            f"baseline={baseline_value:.2f}"
        )


def a3_predict(
    state,
    category=None,
):
    policy_a3.eval()

    with torch.no_grad():
        probabilities = policy_a3(
            torch.tensor(
                state,
                dtype=torch.float,
            )
        )

    return ACTIONS[
        probabilities.argmax().item()
    ]


a3_results = evaluate_agent(
    a3_predict,
    test_data,
    "A3 REINFORCE+Pretrain",
    noise_level=NOISE,
)

torch.save(
    {
        "model_state": policy_a3.state_dict(),
        "actions": ACTIONS,
        "ground_truth": GROUND_TRUTH,
        "results": a3_results,
        "state_dim": 10,
        "action_dim": len(ACTIONS),
        "noise_level": NOISE,
        "seed": SEED,
    },
    SAVE_PATH_A3,
)

print(f"A3 saved to {SAVE_PATH_A3}")


# 10. COMPARISON TABLE

In [ ]:
print("\n" + "="*65)
print("FINAL COMPARISON — ALL AGENTS")
print("="*65)
print(f"{'Agent':30s} {'Accuracy':12s}")
print(f"{'─'*65}")
 
all_results = [a1_results, a2_results, a3_results]
for r in all_results:
    print(f"  {r['label']:28s} {r['accuracy']:8.1f}%   ")
 

# 11. PER-FAULT COMPARISON TABLE

In [ ]:
print("\n" + "="*75)
print("PER-FAULT ACCURACY COMPARISON")
print("="*75)
print(f"{'Fault':6s} {'Expected':15s} {'A1':8s} {'A2':8s} {'A3':8s}")
print(f"{'─'*75}")
 
all_cats = sorted(set(
    list(a1_results['per_fault'].keys()) +
    list(a2_results['per_fault'].keys()) +
    list(a3_results['per_fault'].keys())
))
 
for cat in all_cats:
    exp  = GROUND_TRUTH.get(cat, '?')
    a1   = a1_results['per_fault'].get(cat, 0)
    a2   = a2_results['per_fault'].get(cat, 0)
    a3   = a3_results['per_fault'].get(cat, 0)
    best = max(a1, a2, a3)
    flag = "PASS" if a3 >= 90 else "MARGINAL" if a3 >= 70 else "FAIL"
    print(f"  {flag} {cat:5s}  {exp:13s}  "
          f"{a1:6.1f}%  {a2:6.1f}%  {a3:6.1f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('RL Agent Comparison: A1 vs A2 vs A3', fontsize=14, fontweight='bold')

agents  = ['A1\nRule-based', 'A2\nREINFORCE', 'A3\nPretrain+RL']
colors  = ['#95a5a6', '#3498db', '#2ecc71']
acc     = [a1_results['accuracy'], a2_results['accuracy'], a3_results['accuracy']]

# Chart 1 — Overall accuracy
ax1 = axes[0]
bars = ax1.bar(agents, acc, color=colors, edgecolor='white', linewidth=1.5)
ax1.set_ylabel('Action Accuracy (%)', fontsize=11)
ax1.set_title('Action Accuracy', fontsize=12, fontweight='bold')
ax1.set_ylim(0, 110)
ax1.axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% target')
for bar, val in zip(bars, acc):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.legend(fontsize=9)

# Chart 2 — Per-fault accuracy
ax2 = axes[1]
cats  = sorted(a3_results['per_fault'].keys())
x     = np.arange(len(cats))
width = 0.25

ax2.bar(x - width, [a1_results['per_fault'].get(c,0) for c in cats],
        width, label='A1', color='#95a5a6', edgecolor='white')
ax2.bar(x,         [a2_results['per_fault'].get(c,0) for c in cats],
        width, label='A2', color='#3498db', edgecolor='white')
ax2.bar(x + width, [a3_results['per_fault'].get(c,0) for c in cats],
        width, label='A3', color='#2ecc71', edgecolor='white')

ax2.set_xlabel('Fault Category', fontsize=10)
ax2.set_ylabel('Accuracy (%)', fontsize=11)
ax2.set_title('Per-Fault Accuracy (A1 vs A2 vs A3)', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(cats, rotation=45, fontsize=8)
ax2.set_ylim(0, 115)
ax2.axhline(y=90, color='gray', linestyle='--', alpha=0.4)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('rl_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to rl_comparison.png")


In [ ]:
# Reward Convergence Chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RL Training Convergence: A2 vs A3', fontsize=14, fontweight='bold')
ep_acc_pretrain = a3_results['accuracy']

def rolling(data, window=WINDOW):
    return [np.mean(data[max(0,i-window):i+1]) for i in range(len(data))]

# Chart 1 — Reward convergence
ax1 = axes[0]
ax1.plot(rolling(ep_rewards_a2), color='#3498db', label='A2 REINFORCE', linewidth=1.5, alpha=0.8)
ax1.plot(rolling(ep_rewards_a3), color='#2ecc71', label='A3 Pretrain+RL', linewidth=1.5)
ax1.axhline(y=10, color='gray', linestyle='--', alpha=0.4, label='Max reward (+10)')
ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel(f'Rolling Avg Reward (window={WINDOW})', fontsize=11)
ax1.set_title('Reward Convergence', fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# Chart 2 — Accuracy convergence
ax2 = axes[1]
a2_acc_roll = [v * 100 for v in rolling(ep_correct_a2)]
a3_acc_roll = [v * 100 for v in rolling(ep_correct_a3)]

ax2.plot(a2_acc_roll, color='#3498db', label='A2 REINFORCE', linewidth=1.5, alpha=0.8)
ax2.plot(a3_acc_roll, color='#2ecc71', label='A3 RL phase', linewidth=1.5)
ax2.axhline(y=ep_acc_pretrain, color='#f39c12', linestyle='--', linewidth=1.5,
            label=f'A3 pretrain baseline ({ep_acc_pretrain:.1f}%)')
ax2.axhline(y=90, color='gray', linestyle=':', alpha=0.5, label='90% target')
ax2.set_xlabel('Episode', fontsize=11)
ax2.set_ylabel(f'Rolling Avg Accuracy % (window={WINDOW})', fontsize=11)
ax2.set_title('Accuracy Convergence', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('rl_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print("Convergence chart saved to rl_convergence.png")


In [ ]:
print(policy_a3)